# Open Telemetry (OTEL)

In [10]:
#%pip install -r ../requirements.txt

import os

from dotenv import load_dotenv

import anthropic
import phoenix as px

from phoenix.otel import register

In [11]:
session = px.launch_app()

load_dotenv()
model_name = os.getenv("MODEL_NAME") or ""
message = os.getenv("MESSAGE") or ""

tracer_provider = register(project_name="baseline", auto_instrument=True)

Existing running Phoenix instance detected! Shutting it down and starting a new instance...
⚠️ PHOENIX_COLLECTOR_ENDPOINT is set to http://localhost:6006.
⚠️ This means that traces will be sent to the collector endpoint and not this app.
⚠️ If you would like to use this app to view traces, please unset this environmentvariable via e.g. `del os.environ['PHOENIX_COLLECTOR_ENDPOINT']` 
⚠️ You will need to restart your notebook to apply this change.
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3
Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
DependencyConflict: requested: "crewai >= 1.10.1" but found: "crewai 0.126.0"


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: baseline
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Anthropic SDK

In [12]:

client = anthropic.Anthropic()

response = client.messages.create(
    model=model_name,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": message}
    ]
)

response

Message(id='msg_01PijSocx7X7Xyo7rYMnqQYf', container=None, content=[TextBlock(citations=None, text='## Security Vulnerability Analysis\n\n### Code Reviewed\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## Vulnerabilities Found\n\n### 🔴 Critical Issues\n\n#### 1. **Client-Side Trust / Privilege Escalation**\n```javascript\n// PROBLEM: `user.is_admin` likely comes from user-controlled input\n// Attacker can simply modify:\nuser.is_admin = true; // Browser console, intercepted request, tampered JWT\n```\n**Risk:** Any user can grant themselves admin access\n\n#### 2. **No Authentication Before Authorization**\n```javascript\n// PROBLEM: Checks role without verifying identity first\n// Is this actually the user they claim to be?\n// Missing: token validation, session verification, MFA check\n```\n\n#### 3. **Missing Null/Undefined Check**\n```javascript\n// PROBLEM: If user object is null/undefined → runtime error\n// Could cause denial of service or unexpecte

## Claude Agent SDK

In [13]:
from claude_agent_sdk import query, ClaudeAgentOptions

responses = []
async for msg in query(
    prompt=message,
    options=ClaudeAgentOptions(model=model_name, max_turns=1)
):
    responses.append(msg)

responses

I0330 20:45:07.776728 1577127 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0330 20:45:07.785016 1586337 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(116, generation: 1)
I0330 20:45:07.785053 1586337 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(115, generation: 1)
I0330 20:45:07.785055 1586337 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)
I0330 20:45:07.825859 1577127 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0330 20:45:07.830498 1586378 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(116, generation: 1)
I0330 20:45:07.830521 1586378 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(115, generation: 1)
I0330 20:45:07.830522 1586378 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(104, generation: 1)


[SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/Users/svespie/Code/agent-telemetry-experiment/src', 'session_id': '2349f16b-9b6a-4844-8016-d933696b0c7e', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'NotebookEdit', 'Read', 'RemoteTrigger', 'Skill', 'TaskOutput', 'TaskStop', 'TodoWrite', 'ToolSearch', 'WebFetch', 'WebSearch', 'Write'], 'mcp_servers': [{'name': 'claude.ai Google Calendar', 'status': 'needs-auth'}, {'name': 'claude.ai Gmail', 'status': 'needs-auth'}], 'model': 'claude-sonnet-4-6', 'permissionMode': 'default', 'slash_commands': ['update-config', 'debug', 'simplify', 'batch', 'loop', 'schedule', 'claude-api', 'compact', 'context', 'cost', 'heapdump', 'init', 'pr-comments', 'release-notes', 'review', 'security-review', 'insights'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.87', 'output_styl

## LangChain

In [14]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model=model_name)
response = llm.invoke(message)

response

AIMessage(content='## Security Vulnerability Analysis\n\n### Code Reviewed\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## Vulnerabilities Identified\n\n### 🔴 Critical Issues\n\n#### 1. **Missing Authentication Check**\nThe code checks *authorization* (role) but not *authentication* (identity verification).\n```javascript\n// PROBLEM: Who confirmed this user is who they claim to be?\nif (user.is_admin) { // is_admin could be user-supplied/tampered data\n    session.open();\n}\n```\n\n#### 2. **Privilege Escalation Risk**\nIf `user` object comes from client-side input or an untrusted source, an attacker can manipulate `is_admin`.\n```javascript\n// Attack example:\nconst user = JSON.parse(request.body); // { "is_admin": true }\nif (user.is_admin) { // Trivially bypassed\n    session.open();\n}\n```\n\n#### 3. **No Input Validation**\n`is_admin` has no type enforcement — truthy values could unintentionally pass:\n```javascript\nuser.is_admin = 1;        // 

## Pydantic AI

In [15]:
from pydantic_ai import Agent as PydanticAgent

agent = PydanticAgent(f"anthropic:{model_name}")
result = await agent.run(message)

result

AgentRunResult(output='## Security Vulnerability Analysis\n\n### Code Reviewed\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n## Vulnerabilities Identified\n\n### 🔴 Critical Issues\n\n**1. Missing Authentication Verification**\n```javascript\n// PROBLEM: Only checks a property, not verified identity\nif (user.is_admin) { ... }\n\n// SAFER: Verify through trusted authentication source\nif (authService.verifyAdminRole(user.id, session.token)) { ... }\n```\n\n**2. Client-Side / Untrusted Data Trust**\n- `user.is_admin` may come from **user-controlled input**\n- An attacker could manipulate the object property directly\n- No cryptographic verification of the claim\n\n**3. No Authentication Before Authorization**\n```javascript\n// Missing: Is this user even authenticated first?\nif (!user.isAuthenticated) throw new AuthError();\nif (!authService.verifyAdminRole(user.id)) throw new AuthzError();\nsession.open();\n```\n\n---\n\n### 🟡 Medium Issues\n\n| Issue | Ri

## CrewAI

In [16]:
from crewai import Agent as CrewAgent, Task, Crew, LLM

llm = LLM(model=model_name)

agent = CrewAgent(
    role="Assistant",
    goal="Respond to the user's message",
    backstory="You are a helpful assistant.",
    llm=llm
)

task = Task(
    description=message,
    expected_output="A response to the user's message",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task])
result = crew.kickoff()

result

CrewOutput(raw='## Security Vulnerability Analysis\n\n**Code Under Review:**\n```javascript\nif (user.is_admin) {\n    session.open();\n}\n```\n\n---\n\n### 🔴 Identified Vulnerabilities\n\n---\n\n#### 1. **Insufficient Authentication / Missing Verification of Identity**\n- **Issue:** The code only checks `user.is_admin` — a client-side or untrusted property — without verifying *who* the user actually is (authentication) before checking *what they can do* (authorization).\n- **Risk:** If `user` is a mutable object that can be tampered with (e.g., via JavaScript prototype pollution, deserialization attacks, or direct client manipulation), an attacker could set `is_admin = true` trivially.\n- **Fix:** Always validate identity server-side before trusting role properties.\n\n---\n\n#### 2. **Client-Side Trust / Improper Authorization**\n- **Issue:** `user.is_admin` appears to be a flat boolean flag. If this value is derived from client-supplied input (e.g., a JWT claim that is not verified,

## Compare Traces

Pull all spans from Phoenix and compare what each instrumentor captured.

In [17]:
import time
import pandas as pd

# give the collector a moment to flush
time.sleep(2)

client = px.Client()
df = client.get_spans_dataframe(project_name="baseline")

print(f"Total spans captured: {len(df)}")
print(f"\nColumns available ({len(df.columns)}):")
for col in sorted(df.columns):
    print(f"  {col}")

Total spans captured: 12

Columns available (41):
  attributes.agent_fingerprint
  attributes.crew_agents
  attributes.crew_fingerprint
  attributes.crew_fingerprint_created_at
  attributes.crew_id
  attributes.crew_key
  attributes.crew_memory
  attributes.crew_number_of_agents
  attributes.crew_number_of_tasks
  attributes.crew_process
  attributes.crew_tasks
  attributes.crewai_version
  attributes.input.mime_type
  attributes.input.value
  attributes.llm.input_messages
  attributes.llm.invocation_parameters
  attributes.llm.model_name
  attributes.llm.output_messages
  attributes.llm.provider
  attributes.llm.system
  attributes.llm.token_count.completion
  attributes.llm.token_count.prompt
  attributes.metadata
  attributes.openinference.span.kind
  attributes.output.mime_type
  attributes.output.value
  attributes.python_version
  attributes.task_fingerprint
  attributes.task_fingerprint_created_at
  attributes.task_id
  attributes.task_key
  context.span_id
  context.trace_id
  

/var/folders/8j/f_9x77cd28703_l8b6tlflcw0000gn/T/ipykernel_45429/2486112526.py:8: DeprecationWarning: Migrate to client.spans.get_spans_dataframe() from arize-phoenix-client
  df = client.get_spans_dataframe(project_name="baseline")


In [18]:
# Show all spans with key identifying info
# Use whatever timing column exists
latency_col = next((c for c in df.columns if "latency" in c.lower() or "duration" in c.lower()), None)
cols = ["name", "span_kind", "status_code"]
if latency_col:
    cols.append(latency_col)
df[cols].reset_index(drop=True)

,name,span_kind,status_code
0,Task Created,UNKNOWN,OK
1,Crew Created,UNKNOWN,OK
2,beta.messages.create,LLM,OK
3,messages.create,LLM,OK
4,ChatAnthropic,LLM,OK
5,messages.create,LLM,OK
6,Task Created,UNKNOWN,OK
7,Crew Created,UNKNOWN,OK
8,beta.messages.create,LLM,OK
9,messages.create,LLM,OK


In [19]:
# For each span, show which attribute columns are populated (non-null)
attr_cols = [c for c in df.columns if c.startswith("attributes.")]

records = []
for _, row in df.iterrows():
    populated = [c for c in attr_cols if pd.notna(row[c])]
    records.append({
        "span_name": row["name"],
        "span_kind": row["span_kind"],
        "populated_attributes": populated
    })

for r in records:
    print(f"\n=== {r['span_name']} ({r['span_kind']}) ===")
    for attr in sorted(r["populated_attributes"]):
        print(f"  {attr}")


=== Task Created (UNKNOWN) ===
  attributes.agent_fingerprint
  attributes.crew_fingerprint
  attributes.crew_id
  attributes.crew_key
  attributes.task_fingerprint
  attributes.task_fingerprint_created_at
  attributes.task_id
  attributes.task_key

=== Crew Created (UNKNOWN) ===
  attributes.crew_agents
  attributes.crew_fingerprint
  attributes.crew_fingerprint_created_at
  attributes.crew_id
  attributes.crew_key
  attributes.crew_memory
  attributes.crew_number_of_agents
  attributes.crew_number_of_tasks
  attributes.crew_process
  attributes.crew_tasks
  attributes.crewai_version
  attributes.python_version

=== beta.messages.create (LLM) ===
  attributes.input.mime_type
  attributes.input.value
  attributes.llm.input_messages
  attributes.llm.invocation_parameters
  attributes.llm.model_name
  attributes.llm.output_messages
  attributes.llm.provider
  attributes.llm.system
  attributes.llm.token_count.completion
  attributes.llm.token_count.prompt
  attributes.openinference.span

In [20]:
# Side-by-side: which attributes does each SDK's instrumentor capture?
attr_cols = [c for c in df.columns if c.startswith("attributes.")]

summary = {}
for _, row in df.iterrows():
    name = row["name"]
    populated = {c for c in attr_cols if pd.notna(row[c])}
    if name not in summary:
        summary[name] = populated
    else:
        summary[name] |= populated

# Build a presence matrix
all_attrs = sorted(set().union(*summary.values()))
matrix = pd.DataFrame(
    {span: [attr in attrs for attr in all_attrs] for span, attrs in summary.items()},
    index=all_attrs
).replace({True: "Y", False: ""})

matrix

,Task Created,Crew Created,beta.messages.create,messages.create,ChatAnthropic
attributes.agent_fingerprint,Y,,,,
attributes.crew_agents,,Y,,,
attributes.crew_fingerprint,Y,Y,,,
attributes.crew_fingerprint_created_at,,Y,,,
attributes.crew_id,Y,Y,,,
attributes.crew_key,Y,Y,,,
attributes.crew_memory,,Y,,,
attributes.crew_number_of_agents,,Y,,,
attributes.crew_number_of_tasks,,Y,,,
attributes.crew_process,,Y,,,
